## Hierachical Softmax

### Why?
In multi-class classification, labels often have a hierarchical structure, for example:
```
root
├── A (Sports)
│   ├── A1 (Football)
│   └── A2 (Basketball)
└── B (Finance)
    ├── B1 (Banking)
    └── B2 (Stock Market)
```
Traditional softmax ignores this structure. Hierarchical Softmax (H-Softmax) leverages it to:

- Improve efficiency
- Handle rare subcategories better
- Enhance generalization

Hierarchical Softmax Layers

Every internal node has its own softmax over its children.

To compute P(label = X), chain together probabilities along the path:
$$P(X) = \prod_{n \in \text{path}(\text{root} \rightarrow X)} P(n \mid \text{parent}(n))$$


### 🧩 Components Explained

| Symbol / Term                            | Meaning                                                                 |
|------------------------------------------|-------------------------------------------------------------------------|
| `X`                                      | A **leaf label**, e.g., `A1`, `B2`                                     |
| `path(root → X)`                         | Sequence of nodes from the root to label `X`, e.g., `["root", "A", "A1"]` |
| `n ∈ path(...)`                          | Iterate over each node `n` in the path                                 |
| `P(n | parent(n))`                       | Probability of choosing node `n` given its parent node                 |
| `∏` (product symbol)                     | Multiply all conditional probabilities along the path                  |


- Unified Training Objective
- Instead of separate classifiers at each level, train a single global network that:
- Takes input representation (e.g. from an RNN or transformer encoder).
- Uses shared hidden layers.
- Routes predictions through the appropriate hierarchical softmax module based on label.


$$ P(\text{Football})= P(\text{Sports} \mid root) \times P(\text{Football} \mid \text{Sports})$$

$$
P(\text{Sports} \mid \text{root}) = 0.6
$$

$$
P(\text{Football} \mid \text{Sports}) = 0.7
$$

$$
P(\text{Football}) = 0.6 \times 0.7 = 0.42
$$

- If the tree is balanced, all leaf nodes are about the same depth → fair and efficient.

- If the tree is unbalanced, some leaves (classes) are deeper than others → inefficient and unfair.

# 🔄 Softmax Variants and Comparison with Hierarchical Softmax

---

## 🧠 Common Softmax Variants

| Variant                                | Description                                                           | Key Use Case                              |
| -------------------------------------- | --------------------------------------------------------------------- | ----------------------------------------- |
| **Standard Softmax**                   | Computes full distribution over all classes                           | Small-to-moderate class sizes             |
| **Hierarchical Softmax**               | Breaks class space into a tree and computes conditional probabilities | Large label spaces, e.g., NLP, tagging    |
| **Sampled Softmax**                    | Approximates softmax using only a sample of negative classes          | Training speedup for large vocabularies   |
| **Noise-Contrastive Estimation (NCE)** | Trains model to distinguish real data from noise                      | Word embeddings (Word2Vec, etc.)          |
| **Adaptive Softmax**                   | Allocates more computation to frequent classes, less to rare ones     | Language modeling with large vocabularies |
| **Sparsemax**                          | Produces sparse probability distributions (some values exactly zero)  | Attention, interpretability               |

---

## 📘 Detailed Explanations

### 1. **Standard Softmax**

Computes the full softmax over all classes:

```math
P(y_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}
```

* ✅ Exact
* ❌ Slow with many classes

---

### 2. **Hierarchical Softmax (H-Softmax)**

Uses a tree structure and factorizes the probability along a path:

```math
P(X) = \prod_{n \in \text{path(root} \rightarrow X)} P(n \mid \text{parent}(n))
```

* ✅ Efficient for large label sets
* ✅ Leverages label hierarchy
* ❌ Slightly approximate if tree is unbalanced

---

### 3. **Sampled Softmax**

Approximates softmax by sampling negative classes:

```math
P(y) \approx \frac{e^{z_y}}{e^{z_y} + \sum_{k \in \text{neg samples}} e^{z_k}}
```

* ✅ Faster training
* ❌ Not an exact probability

---

### 4. **Noise-Contrastive Estimation (NCE)**

Trains a binary classifier to distinguish true classes from sampled "noise" classes.

* Common in **Word2Vec skip-gram**
* ❌ Not a probability model

---

### 5. **Adaptive Softmax**

Uses a frequency-based class hierarchy:

* High-frequency classes get more computation

* Low-frequency classes are grouped

* ✅ Efficient

* ✅ Better than sampling in language models

---

### 6. **Sparsemax**

Projects logits onto a probability simplex producing sparse outputs:

```math
\text{Sparsemax}(z) = \arg\min_p \|p - z\|^2 \quad \text{subject to} \quad p \in \Delta^K
```

* ✅ Outputs with exact zeros
* ✅ Useful in interpretable attention

---

## 📊 Comparison Table

| Feature                   | Standard Softmax | Hierarchical Softmax              | Sampled Softmax | Adaptive Softmax  | Sparsemax |
| ------------------------- | ---------------- | --------------------------------- | --------------- | ----------------- | --------- |
| Complexity                | O(N)             | O(log N)                          | O(k)            | O(log N)          | O(N)      |
| Exact Probability         | ✅ Yes            | ✅ Yes (approximate if unbalanced) | ❌ No            | ✅ Mixed           | ✅ Yes     |
| Suitable for Huge Labels  | ❌ No             | ✅ Yes                             | ✅ Yes           | ✅ Yes             | ❌ No      |
| Good for Interpretability | ❌ No             | ✅ Some                            | ❌ No            | ❌ No              | ✅ Yes     |
| Common Usage              | Classification   | Word embeddings, Tagging          | NLP models      | Language Modeling | Attention |

---

## ✅ When to Use What?

| Scenario                                | Best Softmax Variant   |
| --------------------------------------- | ---------------------- |
| < 1,000 classes                         | Standard Softmax       |
| > 10,000 structured classes             | Hierarchical Softmax   |
| Huge vocabularies, training speed focus | Sampled Softmax or NCE |
| Long-tail label distribution            | Adaptive Softmax       |
| Need sparsity or interpretability       | Sparsemax              |

---

Let me know if you’d like:

* Code examples
* Visualization of H-Softmax tree
* A walkthrough on implementing each in PyTorch

Happy to expand any section!


| Feature                   | Standard Softmax | Hierarchical Softmax              | Sampled Softmax | Adaptive Softmax  | Sparsemax |
| ------------------------- | ---------------- | --------------------------------- | --------------- | ----------------- | --------- |
| Complexity                | O(N)             | O(log N)                          | O(k)            | O(log N)          | O(N)      |
| Exact Probability         | ✅ Yes            | ✅ Yes (approximate if unbalanced) | ❌ No            | ✅ Mixed           | ✅ Yes     |
| Suitable for Huge Labels  | ❌ No             | ✅ Yes                             | ✅ Yes           | ✅ Yes             | ❌ No      |
| Good for Interpretability | ❌ No             | ✅ Some                            | ❌ No            | ❌ No              | ✅ Yes     |
| Common Usage              | Classification   | Word embeddings, Tagging          | NLP models      | Language Modeling | Attention |
